In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn


In [2]:
from google.colab import files
uploaded = files.upload()


Saving cleaned_geo_df.parquet to cleaned_geo_df.parquet


In [3]:
import pandas as pd

filename = list(uploaded.keys())[0]
df = pd.read_parquet(filename)

df.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_min,pickup_cluster
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,...,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0,19.800000,0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,...,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,6.600000,0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,...,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0,17.916667,0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,...,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0,8.300000,0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,...,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0,6.100000,0


In [4]:
df['fare_per_mile'] = df['fare_amount'] / df['trip_distance']
df['fare_per_min']  = df['fare_amount'] / df['trip_duration_min']
airport_zones = [132, 138, 141]
df['airport_flag'] = df['PULocationID'].isin(airport_zones).astype(int)
df['hour_of_day'] = df['tpep_pickup_datetime'].dt.hour
df['day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek
df['month']       = df['tpep_pickup_datetime'].dt.month
df = df[
    (df['fare_amount'] > 0) &
    (df['trip_distance'] > 0) &
    (df['trip_duration_min'] > 0)
]
df[['fare_per_mile', 'fare_per_min', 'airport_flag', 'hour_of_day']].head()


,fare_per_mile,fare_per_min,airport_flag,hour_of_day
0,10.290698,0.893939,0,0
1,5.555556,1.515152,0,0
2,4.957447,1.300465,0,0
3,7.142857,1.204819,0,0
4,9.875000,1.295082,0,0


In [5]:
features = [
    'trip_distance',
    'trip_duration_min',
    'passenger_count',
    'VendorID',
    'airport_flag',
    'pickup_cluster',
    'hour_of_day',
    'day_of_week'
]
target = 'fare_amount'


In [6]:
X = df[features]
y = df[target]


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
# Recreate X and y from scratch
features = [
    'trip_distance',
    'trip_duration_min',
    'passenger_count',
    'VendorID',
    'airport_flag',
    'pickup_cluster',
    'hour_of_day',
    'day_of_week'
]
target = 'fare_amount'

X = df[features].copy()
y = df[target].copy()

print("NaNs per feature:")
print(X.isna().sum())

# Drop any rows that have NaNs in X or y
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

print("\nRemaining rows after dropping NaNs:", len(X))


NaNs per feature:
trip_distance             0
trip_duration_min         0
passenger_count      115195
VendorID                  0
airport_flag              0
pickup_cluster            0
hour_of_day               0
day_of_week               0
dtype: int64

Remaining rows after dropping NaNs: 2754407


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

mse, rmse, r2


(193.78269002923855, 13.920585118063054, 0.3477617976091216)

In [10]:
coef_table = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_
})
coef_table


,feature,coefficient
0,trip_distance,0.368806
1,trip_duration_min,0.092964
2,passenger_count,0.565668
3,VendorID,0.744427
4,airport_flag,25.495139
5,pickup_cluster,7.467406
6,hour_of_day,-0.032254
7,day_of_week,-0.109819
